# 01 — Perception API Wrapper
**Purpose:** Simulate a greybox MLaaS perception API backed by YOLO11n.  
**Output:** `data/query_log.jsonl` — one line per query, used by the monitor later.  
**Supports:** Single image, batch images, video file, live webcam.

## 1. Setup

In [3]:
import hashlib
import json
import time
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO

# Paths
LOG_FILE = Path("../data/query_log.jsonl")
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

# Load model once
model = YOLO("../models/yolo11n.pt")

# Global query counter (resets each session — that's fine for now)
query_counter = 0

print(f"Model loaded. Log → {LOG_FILE.resolve()}")

Model loaded. Log → /JUNK/Work-things/GIT/YOLO-Monitoring/data/query_log.jsonl


## 2. Core API Function

In [5]:
def query(image, source_label="unknown"):
    """
    Simulated perception API.

    Args:
        image: file path (str/Path) OR numpy array (BGR, from cv2)
        source_label: tag for log ('image', 'video', 'webcam', 'attack_knockoff', etc.)

    Returns:
        list of dicts: [{class, class_name, conf, bbox}, ...]
        Empty list if no detections.
    """
    global query_counter
    query_counter += 1

    # --- Image fingerprint (for dedup / pattern analysis) ---
    if isinstance(image, (str, Path)):
        image_hash = hashlib.md5(Path(image).read_bytes()).hexdigest()
    else:
        image_hash = hashlib.md5(image.tobytes()).hexdigest()

    # --- Run inference ---
    t0 = time.perf_counter()
    results = model(image, verbose=False)
    latency_ms = round((time.perf_counter() - t0) * 1000, 1)

    r = results[0]

    # --- Format response (what attacker receives) ---
    detections = []
    for box in r.boxes:
        detections.append({
            "class":      int(box.cls[0]),
            "class_name": model.names[int(box.cls[0])],
            "conf":       round(float(box.conf[0]), 4),
            "bbox":       [round(x, 2) for x in box.xyxy[0].tolist()]  # [x1, y1, x2, y2]
        })

    # --- Log entry (what monitor reads) ---
    entry = {
        "query_id":    query_counter,
        "timestamp":   time.time(),
        "source":      source_label,
        "image_hash":  image_hash,
        "n_detections": len(detections),
        "latency_ms":  latency_ms,
        "detections":  detections
    }

    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")

    return detections

## 3. Mode A — Single Image

In [6]:
IMAGE_PATH = "../data/gettyimage.jpg"  # ← change to your image

result = query(IMAGE_PATH, source_label="image")

print(f"Detections: {len(result)}")
for d in result:
    print(f"  [{d['class_name']:15s}] conf={d['conf']:.2f}  bbox={d['bbox']}")

Detections: 13
  [car            ] conf=0.92  bbox=[200.56, 236.36, 418.58, 394.01]
  [car            ] conf=0.88  bbox=[600.66, 222.7, 743.52, 334.95]
  [truck          ] conf=0.80  bbox=[70.72, 147.44, 239.95, 311.38]
  [car            ] conf=0.77  bbox=[0.04, 205.62, 172.37, 348.07]
  [car            ] conf=0.76  bbox=[318.65, 217.27, 446.78, 332.31]
  [car            ] conf=0.74  bbox=[0.06, 278.16, 35.98, 377.7]
  [car            ] conf=0.70  bbox=[487.31, 199.94, 530.01, 243.14]
  [car            ] conf=0.69  bbox=[450.7, 205.98, 502.84, 282.91]
  [car            ] conf=0.66  bbox=[352.19, 196.0, 471.02, 300.7]
  [car            ] conf=0.62  bbox=[571.1, 219.64, 616.74, 269.71]
  [car            ] conf=0.39  bbox=[226.03, 197.45, 301.41, 249.14]
  [car            ] conf=0.31  bbox=[277.13, 190.97, 336.83, 237.96]
  [car            ] conf=0.27  bbox=[575.13, 199.51, 641.76, 229.77]


## 4. Mode B — Video File

In [10]:
VIDEO_PATH = "../data/dash-cam-video.mp4"   # ← change to your video
MAX_FRAMES = 5000                        # limit for testing; set None for full video
SAVE_OUTPUT = True                      # save annotated video?
OUTPUT_PATH = "../data/video-output_annotated.mp4"

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open: {VIDEO_PATH}")

fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
limit  = min(MAX_FRAMES, total) if MAX_FRAMES else total

print(f"Video: {width}x{height} @ {fps:.1f}fps | Processing {limit}/{total} frames")

writer = None
if SAVE_OUTPUT:
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

frame_idx = 0
while frame_idx < limit:
    ret, frame = cap.read()
    if not ret:
        break

    # Query API — frame is numpy array (BGR)
    detections = query(frame, source_label="video")

    # Annotate frame
    for d in detections:
        x1, y1, x2, y2 = [int(v) for v in d["bbox"]]
        label = f"{d['class_name']} {d['conf']:.2f}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, label, (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    if writer:
        writer.write(frame)

    frame_idx += 1
    if frame_idx % 10 == 0:
        print(f"  Frame {frame_idx}/{limit}", end="\r")

cap.release()
if writer:
    writer.release()

print(f"\nDone. Processed {frame_idx} frames.")
if SAVE_OUTPUT:
    print(f"Annotated video → {OUTPUT_PATH}")

Video: 640x360 @ 30.0fps | Processing 5000/18542 frames
  Frame 5000/5000
Done. Processed 5000 frames.
Annotated video → ../data/video-output_annotated.mp4


## 5. Mode C — Webcam (live)

In [ ]:
# Press Q in the OpenCV window to stop
# NOTE: requires a display (won't work headless)

cap = cv2.VideoCapture(0)  # 0 = default webcam
print("Webcam running. Press Q to stop.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    detections = query(frame, source_label="webcam")

    for d in detections:
        x1, y1, x2, y2 = [int(v) for v in d["bbox"]]
        label = f"{d['class_name']} {d['conf']:.2f}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 200, 255), 2)
        cv2.putText(frame, label, (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 255), 1)

    cv2.imshow("Perception API — Live", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
print(f"Session ended. Total queries this session: {query_counter}")

## 6. Inspect the Query Log

In [11]:
# Read last N entries from log
N = 5

with open(LOG_FILE) as f:
    lines = f.readlines()

print(f"Total logged queries: {len(lines)}")
print(f"\nLast {N} entries:")
for line in lines[-N:]:
    entry = json.loads(line)
    print(f"  qid={entry['query_id']:4d} | source={entry['source']:10s} | "
          f"n_det={entry['n_detections']:2d} | latency={entry['latency_ms']}ms | "
          f"hash={entry['image_hash'][:8]}...")

Total logged queries: 8830

Last 5 entries:
  qid=8827 | source=video      | n_det= 5 | latency=4.1ms | hash=0484bd99...
  qid=8828 | source=video      | n_det= 6 | latency=4.1ms | hash=6797df85...
  qid=8829 | source=video      | n_det= 5 | latency=3.8ms | hash=c77e85d7...
  qid=8830 | source=video      | n_det= 6 | latency=4.4ms | hash=1d4e33a9...
  qid=8831 | source=video      | n_det= 6 | latency=4.1ms | hash=2dce1077...
